In [1]:
import torch
from kernels.loader import naive_attn

In [2]:
Q = torch.randn(size = (32,8,10,64))
K = torch.randn(size = (32,8,10,64))
V = torch.randn(size = (32,8,10,64))
scaling = 1 / (64 ** 0.5)

In [3]:
result = naive_attn(Q,K,V,scaling)

In [4]:
result.shape

torch.Size([32, 8, 10, 64])

In [5]:
def verify_parity():
    # 1. Setup dimensions (keep them small for initial debug)
    B, H, N, D = 2, 4, 64, 32
    scaling = 1.0 / (D ** 0.5)
    device = torch.device("cuda")

    # 2. Initialize Tensors
    # Use float32 to match your kernel's float* pointers
    q = torch.randn(B, H, N, D, device=device, dtype=torch.float32)
    k = torch.randn(B, H, N, D, device=device, dtype=torch.float32)
    v = torch.randn(B, H, N, D, device=device, dtype=torch.float32)

    # 3. PyTorch Reference (The "Ground Truth")
    # Step-by-step to match your kernel's loop logic
    attn_weights = torch.matmul(q, k.transpose(-2, -1)) * scaling
    ref_out = torch.matmul(attn_weights, v)

    # 4. Your CUDA Kernel
    # Ensure your loader.naive_attn calls the C++ backend correctly
    custom_out = naive_attn(q, k, v, scaling)

    # 5. Numerical Parity Check
    # atol (absolute tolerance) and rtol (relative tolerance) 
    # are needed because floating point sum order differs
    is_correct = torch.allclose(ref_out, custom_out, atol=1e-5, rtol=1e-5)
    
    max_diff = (ref_out - custom_out).abs().max().item()

    if is_correct:
        print(f"✅ SUCCESS: Numerical Parity Achieved!")
        print(f"Max Absolute Difference: {max_diff:.2e}")
    else:
        print(f"❌ FAILURE: Results mismatch!")
        print(f"Max Absolute Difference: {max_diff:.2e}")
        # Print a small slice for debugging
        print("\nReference (First 4 elements of head 0):")
        print(ref_out[0, 0, 0, :4])
        print("\nCustom Kernel (First 4 elements of head 0):")
        print(custom_out[0, 0, 0, :4])

In [6]:
verify_parity()

✅ SUCCESS: Numerical Parity Achieved!
Max Absolute Difference: 7.63e-06
